# LLMs as Persuasion Engines
## P@S Lecture 23: Can AI write the messages?

The cost of generating a unique persuasive message for each person
has dropped from infinite (hire a speechwriter) to near zero (one API call).
Today: what does the evidence say about LLM persuasion?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

BLUE = '#2980b9'
RED = '#c0392b'
GREEN = '#27ae60'
ORANGE = '#e67e22'
PURPLE = '#8e44ad'
GRAY = '#7f8c8d'

## Section 1: The scale of persuasion research

How has persuasion research evolved over time?

In [ ]:
# Evolution of persuasion research
studies = pd.DataFrame({
    'era': ['Aristotle\n(~350 BCE)', 'Focus groups\n(1940s)', 'TV ads\nCoppock (2016)',
            'Platform A/B\nBond (2010)', 'LLM dialogue\nHackenburg (2025)'],
    'n_participants': [50, 30, 34000, 61000000, 76977],
    'messages_per_person': [1, 1, 1, 1, 1],  # For the bar chart
    'cost_per_message': [10000, 1000, 100, 0.01, 0.001],
    'unique_messages': [1, 1, 59, 3, 707],
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: sample sizes (log scale)
bars = ax1.barh(studies['era'], studies['n_participants'], color=BLUE)
ax1.set_xscale('log')
ax1.set_xlabel('Number of Participants (log scale)')
ax1.set_title('Scale of Persuasion Research Over Time')

# Annotate
for i, (era, n) in enumerate(zip(studies['era'], studies['n_participants'])):
    if n >= 1000000:
        label = f'{n/1000000:.0f}M'
    elif n >= 1000:
        label = f'{n/1000:.0f}K'
    else:
        label = str(n)
    ax1.text(n * 1.5, i, label, va='center', fontsize=10, fontweight='bold')

# Right: cost per unique message
ax2.barh(studies['era'], studies['cost_per_message'], color=ORANGE)
ax2.set_xscale('log')
ax2.set_xlabel('Cost per Unique Message (USD, log scale)')
ax2.set_title('Cost of Persuasion Has Collapsed')

for i, (era, cost) in enumerate(zip(studies['era'], studies['cost_per_message'])):
    label = f'${cost:,.2f}' if cost < 1 else f'${cost:,.0f}'
    ax2.text(cost * 1.5, i, label, va='center', fontsize=10)

plt.tight_layout()
plt.show()

**What you should see:** Two trends moving in opposite directions.
Sample sizes have exploded (from 50 to 61 million).
Cost per unique message has collapsed (from $10,000 to $0.001).
This combination means personalized persuasion at scale is now economically viable.

## Section 2: Does model scale make models more persuasive?

Hackenburg et al. (2025) tested 19 different LLMs of varying sizes.
The key question: do bigger models persuade better?

*Data below is synthetic, matching the published pattern of diminishing returns.*

In [ ]:
# Synthetic data matching Hackenburg et al. findings
# The key result: sharp diminishing returns from model scale
models = pd.DataFrame({
    'model': ['GPT-2\n(1.5B)', 'LLaMA-7B', 'Mistral-7B', 'LLaMA-13B',
              'GPT-3.5', 'LLaMA-70B', 'Mixtral-8x7B', 'Claude 2',
              'GPT-4', 'Claude 3\nSonnet', 'GPT-4o', 'Claude 3\nOpus'],
    'params_billions': [1.5, 7, 7, 13, 175, 70, 47, 130, 1800, 200, 1800, 500],
    # Persuasion effect (standardized attitude change)
    # Pattern: sharp rise from tiny to medium, then flat
    'persuasion_effect': [0.02, 0.08, 0.09, 0.10, 0.12, 0.11, 0.11, 0.13,
                          0.13, 0.14, 0.14, 0.14],
    'persuasion_se': [0.02, 0.02, 0.02, 0.02, 0.01, 0.02, 0.02, 0.01,
                      0.01, 0.01, 0.01, 0.01],
})

fig, ax = plt.subplots(figsize=(12, 6))
ax.errorbar(models['params_billions'], models['persuasion_effect'],
            yerr=1.96 * models['persuasion_se'],
            fmt='o', color=BLUE, markersize=8, capsize=5, linewidth=1.5)

# Add model labels
for _, row in models.iterrows():
    ax.annotate(row['model'], xy=(row['params_billions'], row['persuasion_effect']),
                xytext=(5, 8), textcoords='offset points', fontsize=8,
                alpha=0.8)

# Logarithmic trend line
from numpy.polynomial import polynomial as P
log_x = np.log(models['params_billions'])
coeffs = np.polyfit(log_x, models['persuasion_effect'], 1)
x_smooth = np.logspace(np.log10(1), np.log10(2000), 100)
y_smooth = np.polyval(coeffs, np.log(x_smooth))
ax.plot(x_smooth, y_smooth, '--', color=RED, alpha=0.5, label='Log trend (diminishing returns)')

ax.set_xscale('log')
ax.set_xlabel('Model Size (billions of parameters, log scale)')
ax.set_ylabel('Persuasion Effect (standardized attitude change)')
ax.set_title('Hackenburg et al. (2025): Bigger Models Are NOT Much More Persuasive\n(Synthetic data matching published pattern)')
ax.legend()
ax.set_ylim(-0.02, 0.22)
plt.tight_layout()
plt.show()

**What you should see:** A curve that rises sharply from tiny models (GPT-2)
to medium models (GPT-3.5), then FLATTENS. GPT-4 and Claude 3 Opus are
barely more persuasive than GPT-3.5 or LLaMA-70B.

**The takeaway:** once a model is coherent enough to stay on topic,
making it bigger barely increases its persuasive power. The fear that
"superintelligent AI will be super-persuasive" is not supported by this data.

## Section 3: What DOES make models more persuasive?

If scale doesn't matter much, what does?

In [ ]:
# From Hackenburg et al.: relative contribution of different factors
factors = ['Post-training\n(RLHF)', 'Prompting\nstrategy', 'Model\nscale', 'Personali-\nzation']
effects_pct = [51, 27, 12, 10]
colors = [RED, ORANGE, BLUE, GRAY]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(factors, effects_pct, color=colors, height=0.5)
ax.set_xlabel('Relative Contribution to Persuasive Effect (%)')
ax.set_title('What Makes an LLM More Persuasive?\n(Hackenburg et al. 2025)')

# Annotate bars
for bar, pct in zip(bars, effects_pct):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{pct}%', va='center', fontweight='bold', fontsize=12)

ax.set_xlim(0, 65)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

**What you should see:** Post-training (RLHF) is the biggest lever (51%),
followed by prompting strategy (27%). Model scale and personalization
together account for only ~22%.

**Translation:** the persuasion risk comes from HOW models are trained
and instructed, not from raw capability. The instruction "be helpful and
engaging" already contains the seed of persuasion. Making the model bigger
adds little. Tailoring the message to the individual adds little.
Training the model to be confident and responsive adds a LOT.

## Section 4: The persuasion-accuracy tradeoff

The most persuasive model outputs are also the least accurate.

In [ ]:
# Simulate the tradeoff: more claims per response = more persuasive, but less accurate
np.random.seed(42)
n_responses = 200

# Information density: how many fact-checkable claims per response
info_density = np.random.uniform(1, 10, n_responses)

# Persuasion increases with density (more claims = more convincing)
persuasion = 0.05 + 0.015 * info_density + np.random.normal(0, 0.02, n_responses)

# Accuracy DECREASES with density (more claims = more chances for errors)
accuracy = 0.95 - 0.04 * info_density + np.random.normal(0, 0.05, n_responses)
accuracy = np.clip(accuracy, 0.3, 1.0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: persuasion vs. accuracy
ax1.scatter(accuracy, persuasion, alpha=0.4, color=PURPLE, s=30)
# Trend line
z = np.polyfit(accuracy, persuasion, 1)
x_line = np.linspace(accuracy.min(), accuracy.max(), 100)
ax1.plot(x_line, np.polyval(z, x_line), '--', color=RED, linewidth=2)
ax1.set_xlabel('Factual Accuracy (% of claims that are true)')
ax1.set_ylabel('Persuasion Effect')
ax1.set_title('The Tradeoff: More Persuasive = Less Accurate')

# Right: both vs. information density
ax2.scatter(info_density, persuasion, alpha=0.4, color=GREEN, s=30, label='Persuasion')
ax2_twin = ax2.twinx()
ax2_twin.scatter(info_density, accuracy, alpha=0.4, color=RED, s=30, label='Accuracy')
ax2.set_xlabel('Information Density (fact-checkable claims per response)')
ax2.set_ylabel('Persuasion Effect', color=GREEN)
ax2_twin.set_ylabel('Factual Accuracy', color=RED)
ax2.set_title('Why the Tradeoff Exists:\nMore Claims = More Persuasive, but Less Accurate')

# Combined legend
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.suptitle('Hackenburg et al. (2025): Persuasion and Accuracy Move in Opposite Directions',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**What you should see:** A negative correlation between persuasion and accuracy.
Responses packed with claims are more persuasive (people find them convincing)
but less accurate (more chances for fabrication). The model learns that
confident, specific, evidence-rich responses work best. But some of that
"evidence" is invented.

## Section 5: Costello's conspiracy debunking

GPT-4 talked 2,190 conspiracy believers into updating their beliefs.
The effect lasted at least 2 months.

In [ ]:
# Simulate Costello et al. findings
time_points = ['Before\ndialogue', 'Immediately\nafter', '10 days\nlater', '2 months\nlater']
belief_control = [80, 79, 78, 79]  # Control group: stable beliefs
belief_treatment = [80, 64, 65, 63]  # Treatment: ~20% reduction, persistent

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(4), belief_control, 'o-', color=GRAY, linewidth=2, markersize=10,
        label='Control (no AI dialogue)')
ax.plot(range(4), belief_treatment, 'o-', color=GREEN, linewidth=2, markersize=10,
        label='Treatment (AI dialogue)')

# Shade the effect
ax.fill_between(range(4), belief_control, belief_treatment, alpha=0.15, color=GREEN)

# Annotate the effect size
ax.annotate(f'~20% reduction\n(persists 2 months)',
            xy=(3, (belief_control[3] + belief_treatment[3]) / 2),
            xytext=(2.2, 85), fontsize=11, fontweight='bold', color=GREEN,
            arrowprops=dict(arrowstyle='->', color=GREEN))

ax.set_xticks(range(4))
ax.set_xticklabels(time_points)
ax.set_ylabel('Belief Strength (0-100 scale)')
ax.set_title('Costello et al. (2024): GPT-4 Durably Reduces Conspiracy Beliefs')
ax.legend()
ax.set_ylim(50, 95)
plt.tight_layout()
plt.show()

**What you should see:** The green line drops ~20% after the AI dialogue
and STAYS down for 2 months. The gray line (control) stays flat.
Traditional debunking typically shows smaller effects that fade within days.

**The mechanism:** evidence, not empathy. The AI generates specific facts
tailored to each person's particular version of the conspiracy.
A human fact-checker might take 30 minutes; GPT-4 does it in seconds.

## Section 6: Bai's real-election test

Chatbots were deployed during ACTUAL elections (US 2024, Canada 2025, Poland 2025).

In [ ]:
# Compare persuasion effects across modalities
modalities = ['TV ads\n(Coppock 2020)', 'Social media ads\n(Matz 2017)',
              'AI chatbot\n(Bai, US 2024)', 'AI chatbot\n(Bai, Canada 2025)',
              'AI chatbot\n(Bai, Poland 2025)']
effects_sd = [0.05, 0.08, 0.12, 0.18, 0.15]  # Standardized effects (approximate)
colors_bar = [GRAY, BLUE, GREEN, GREEN, GREEN]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(modalities, effects_sd, color=colors_bar, width=0.5)
ax.set_ylabel('Persuasion Effect (standardized)')
ax.set_title('Bai et al. (2025): AI Chatbots Outperform Traditional Persuasion\n'
             'Tested During ACTUAL Elections')

# Annotate
for bar, eff in zip(bars, effects_sd):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{eff:.2f} SD', ha='center', fontsize=10, fontweight='bold')

# Add the fabrication warning
ax.annotate('Chatbots fabricated some\n"evidence" to be more persuasive',
            xy=(3, 0.18), xytext=(3.5, 0.22),
            fontsize=10, color=RED, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=RED))

ax.set_ylim(0, 0.28)
plt.tight_layout()
plt.show()

**What you should see:** AI chatbots (green bars) produce larger effects
than traditional TV or social media ads (gray/blue bars).

**The catch (red annotation):** chatbots were more persuasive when they
cited evidence. But some of that evidence was fabricated by the model.
The most persuasive modality is also the least trustworthy.

## Section 7: The cost calculation

What does it cost to persuade a million people?

In [ ]:
# Cost comparison
methods = ['30-sec TV ad\n(reach 1M viewers)', 'Facebook ad campaign\n(reach 1M users)',
           '1M personalized\nAI chatbot conversations']
costs = [20000, 5000, 1000]
per_person = [0.020, 0.005, 0.001]
unique_msg = ['1 message\n(same for everyone)', '5-10 variants\n(A/B tested)',
              '1,000,000 unique\n(one per person)']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: total cost
ax1.barh(methods, costs, color=[GRAY, BLUE, GREEN], height=0.5)
ax1.set_xlabel('Total Cost (USD)')
ax1.set_title('Cost to Reach 1 Million People')
for i, cost in enumerate(costs):
    ax1.text(cost + 200, i, f'${cost:,}', va='center', fontsize=11, fontweight='bold')

# Right: unique messages
ax2.barh(methods, [1, 7, 1000000], color=[GRAY, BLUE, GREEN], height=0.5)
ax2.set_xscale('log')
ax2.set_xlabel('Number of Unique Messages (log scale)')
ax2.set_title('Message Uniqueness')
for i, um in enumerate(unique_msg):
    ax2.text(10, i, um, va='center', fontsize=9)

plt.suptitle('The Economics of Persuasion Have Changed', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**What you should see:** AI chatbots are the cheapest method AND produce
the most unique messages. For $1,000, you get a million unique, personalized
conversations. A TV ad costs 20x more and shows everyone the same thing.

This is why LLM persuasion matters: the economic constraint on personalized
persuasion at scale is gone.

## Key Takeaways

1. **Bigger models are NOT much more persuasive.** Diminishing returns from scale. GPT-3.5 and GPT-4 are similarly persuasive.

2. **Training and prompting are the real levers.** RLHF (51%) and prompting (27%) drive persuasive power more than scale (12%) or personalization (10%).

3. **Persuasion trades off with accuracy.** The most persuasive outputs contain the most fabricated claims. The model learns that confident, specific claims work. It doesn't learn which claims are true.

4. **The cost of persuasion has collapsed.** A million unique conversations for $1,000. The economic barrier to personalized political persuasion is gone.